# Phase 0 — Setup & GPU Basics

**Run cells top-to-bottom.** Edit code, re-run, break things — that's how you learn.

**Goals:**
- Confirm PyTorch sees your RTX 4070
- Understand `.device`, `.dtype`, `.shape`
- Move tensors CPU ↔ GPU

**Tip:** Run `?torch.Tensor.to` in a cell to pull up docs interactively.

In [1]:
import torch
from shared.utils.device import get_device

print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")

PyTorch : 2.6.0+cu124
CUDA    : True


## 0.1 — The mental model

Every tensor has three questions:

| Property | Question |
|----------|----------|
| `.shape` | How big? |
| `.device` | CPU or GPU? |
| `.dtype` | What precision/type? |

PyTorch code runs in Python; math runs on the tensor's **device**.

### 🧪 Quiz (answer in the next cell)

```python
x = torch.randn(4, 4)              # no device specified
y = torch.randn(4, 4, device="cuda")
```

**Before running:** Where will `x` live? Where will `y` live?

In [2]:
# Run this, then compare to your prediction
x = torch.randn(4, 4)
y = torch.randn(4, 4, device="cuda")

print(f"x → {x.device}")
print(f"y → {y.device}")

x → cpu
y → cuda:0


## 0.2 — GPU check

Your RTX 4070 (Ada, compute cap **8.9**) has **8 GB VRAM** — plenty for learning. Larger models later will use mixed precision (Phase 5).

In [3]:
assert torch.cuda.is_available(), "Install CUDA build: pip install torch --index-url https://download.pytorch.org/whl/cu124"

props = torch.cuda.get_device_properties(0)
print(f"GPU     : {props.name}")
print(f"Compute : {props.major}.{props.minor}")
print(f"VRAM    : {props.total_memory / 1024**3:.1f} GB")

# A real GPU operation
x = torch.randn(512, 512, device="cuda")
y = x @ x.T
print(f"Matmul  : {tuple(y.shape)} on {y.device}")

GPU     : NVIDIA GeForce RTX 4070 Laptop GPU
Compute : 8.9
VRAM    : 8.0 GB
Matmul  : (512, 512) on cuda:0


## 0.3 — CPU ↔ GPU transfers

`.to(device)` **copies** data to a new tensor. The original stays put.

Try changing `"cuda"` to `"cpu"` below and re-run.

In [4]:
device = get_device()
print(f"Default device: {device}")

a = torch.tensor([[1., 2.], [3., 4.]])
print(f"a          : {a.device}")

b = a.to(device)
print(f"b = a.to() : {b.device}")
print(f"a still on : {a.device}")   # ← original unchanged

c = b.cpu()
print(f"c = b.cpu(): {c.device}")

Default device: cuda
a          : cpu
b = a.to() : cuda:0
a still on : cpu
c = b.cpu(): cpu


### 🧪 Explore: what breaks?

Uncomment **one line at a time** in the next cell. Read the error — this is a mistake you'll make often in Phase 3+.

In [8]:
cpu_t = torch.ones(2, 2)
gpu_t = torch.ones(2, 2, device="cuda")

# Uncomment ONE line at a time:
# cpu_t + gpu_t          # device mismatch!
# cpu_t @ gpu_t          # same problem
gpu_t + gpu_t            # ✓ both on GPU — works

tensor([[2., 2.],
        [2., 2.]], device='cuda:0')

## 0.4 — dtypes & memory

`float32` = 4 bytes/element (default). `float16` = 2 bytes — **half the VRAM**.

On an 8 GB GPU, dtype choice matters for large models.

In [9]:
def tensor_mb(t: torch.Tensor) -> float:
    return t.element_size() * t.numel() / 1024**2

n = 1000
f32 = torch.zeros(n, n, dtype=torch.float32)
f16 = torch.zeros(n, n, dtype=torch.float16)

print(f"{n}×{n} float32 : {tensor_mb(f32):.1f} MB")
print(f"{n}×{n} float16 : {tensor_mb(f16):.1f} MB")
print(f"\nf32 dtype: {f32.dtype}, device: {f32.device}")

1000×1000 float32 : 3.8 MB
1000×1000 float16 : 1.9 MB

f32 dtype: torch.float32, device: cpu


## 0.5 — Your turn ✏️

Fill in the cell below:
1. Create `t` — a 3×3 tensor of **ones** on **GPU**, **float32**
2. Create `doubled = t * 2`
3. Print both

<details><summary>Hint (try first without peeking)</summary>

```python
t = torch.ones(3, 3, device="cuda", dtype=torch.float32)
doubled = t * 2
```
</details>

In [10]:
# YOUR CODE HERE
from torch import float32


t = torch.ones(3,3, dtype=float32, device="cuda")
doubled = t * 2
 
print("t:\n", t)
print("\ndoubled:\n", doubled)
print(f"\nshape={t.shape}, device={t.device}, dtype={t.dtype}")

t:
 tensor([[1., 1., 1.],
        [1., 1., 1.],
        [1., 1., 1.]], device='cuda:0')

doubled:
 tensor([[2., 2., 2.],
        [2., 2., 2.],
        [2., 2., 2.]], device='cuda:0')

shape=torch.Size([3, 3]), device=cuda:0, dtype=torch.float32


In [11]:
# Run after your exercise — checks your answer
assert t.shape == (3, 3), "t should be 3×3"
assert t.device.type == "cuda", "t should be on GPU"
assert t.dtype == torch.float32, "t should be float32"
assert torch.allclose(doubled, t * 2), "doubled should equal t * 2"
print("✓ Phase 0 exercise passed!")

✓ Phase 0 exercise passed!


## ✅ Phase 0 complete

You now know:
- **`.device`** — where computation runs
- **`.to(device)`** — copies; doesn't move in-place
- **`.dtype`** — precision & memory tradeoff
- **Device mismatch** — #1 beginner bug (CPU tensor + GPU tensor)

**Next:** `01_foundations/01_tensor_basics.ipynb` — tensor creation, shapes, broadcasting.

Tell your tutor **"Start Phase 1"** when ready.